# Opening the Black Box: PyTorch Hooks into a Running LLM

This notebook loads **Llama 3.1 8B Instruct** on an A100 GPU and attaches forward/backward hooks to inspect activations, attention patterns, gradient flow, and layer-by-layer behavior during inference.

### Steps
1. Load model and tokenizer
2. Attach forward hooks to capture activations
3. Run inference with hooks active
4. Analyze activation statistics
5. Visualize activation norms across layers
6. Bottleneck detection (dead neurons, near-zero std)
7. Layer timing profiling (pre-hook + forward hook)

In [ ]:
import torch
import torch.nn as nn
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import json
import time
import os
from collections import defaultdict
from transformers import AutoModelForCausalLM, AutoTokenizer

%matplotlib inline

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Step 1: Load Model and Tokenizer

Loading Llama 3.1 8B Instruct in FP16. Full model fits comfortably in A100 80GB (~16GB VRAM).

In [ ]:
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
CACHE_DIR = os.path.expanduser("~/models")
OUTPUT_DIR = os.path.expanduser("~/hook_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

PROMPT = "Explain how a transformer model processes text step by step."

print("Loading model and tokenizer...")
t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=CACHE_DIR,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

load_time = time.time() - t0
print("Model loaded in {:.1f}s".format(load_time))
print("Parameters: {:.2f}B".format(sum(p.numel() for p in model.parameters()) / 1e9))
print("Device:", next(model.parameters()).device)
print("Architecture:", model.config.architectures)
print("Layers:", model.config.num_hidden_layers)
print("Hidden size:", model.config.hidden_size)
print("Attention heads:", model.config.num_attention_heads)

## Step 2: Attach Forward Hooks to Capture Activations

We attach a forward hook to **every** decoder layer, plus the embedding layer and final norm. Each hook captures:
- Shape, mean, std, min, max, norm, and dead neuron percentage

In [ ]:
activations = {}
activation_stats = {}

def make_activation_hook(name):
    """Forward hook that captures activation statistics."""
    def hook(module, input, output):
        # Handle tuple outputs (common in transformer layers)
        if isinstance(output, tuple):
            tensor = output[0]
        else:
            tensor = output

        if isinstance(tensor, torch.Tensor):
            activations[name] = tensor.detach().cpu().float()
            activation_stats[name] = {
                "shape": list(tensor.shape),
                "mean": tensor.float().mean().item(),
                "std": tensor.float().std().item(),
                "min": tensor.float().min().item(),
                "max": tensor.float().max().item(),
                "norm": tensor.float().norm().item(),
                "zeros_pct": (tensor == 0).float().mean().item() * 100,
            }
    return hook

# Attach to every decoder layer + embeddings + final norm
handles = []

# Embedding layer
h = model.model.embed_tokens.register_forward_hook(make_activation_hook("embed_tokens"))
handles.append(h)

# Each transformer layer (full layer + attention + MLP sub-modules)
for i, layer in enumerate(model.model.layers):
    h = layer.register_forward_hook(make_activation_hook("layer_{}".format(i)))
    handles.append(h)
    h = layer.self_attn.register_forward_hook(make_activation_hook("layer_{}_attn".format(i)))
    handles.append(h)
    h = layer.mlp.register_forward_hook(make_activation_hook("layer_{}_mlp".format(i)))
    handles.append(h)

# Final layer norm
h = model.model.norm.register_forward_hook(make_activation_hook("final_norm"))
handles.append(h)

print("Attached {} forward hooks".format(len(handles)))

## Step 3: Run Inference with Hooks Active

Now we run `model.generate()` and all hooks fire automatically, capturing data from every layer.

In [ ]:
print('Prompt: "{}"'.format(PROMPT))

inputs = tokenizer(PROMPT, return_tensors="pt").to(model.device)
input_ids = inputs["input_ids"]
print("Input tokens:", input_ids.shape[1])

t0 = time.time()
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        return_dict_in_generate=True,
        output_hidden_states=True,
        output_attentions=True,
    )
gen_time = time.time() - t0

generated_text = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)
new_tokens = outputs.sequences.shape[1] - input_ids.shape[1]
print("Generated {} tokens in {:.1f}s ({:.1f} tok/s)".format(new_tokens, gen_time, new_tokens/gen_time))
print()
print("Response:")
print(generated_text[:500])

## Step 4: Analyze Activation Statistics

Let's look at what each hook captured -- the activation statistics across all layers.

In [ ]:
header = "{:<25} {:<25} {:>8} {:>8} {:>10} {:>10} {:>10} {:>6}".format(
    "Layer", "Shape", "Mean", "Std", "Min", "Max", "Norm", "Dead%")
print(header)
print("-" * len(header))
for name, stats in sorted(activation_stats.items()):
    shape_str = str(stats["shape"])
    print("{:<25} {:<25} {:>8.4f} {:>8.4f} {:>10.4f} {:>10.4f} {:>10.2f} {:>5.1f}%".format(
        name, shape_str, stats["mean"], stats["std"],
        stats["min"], stats["max"], stats["norm"], stats["zeros_pct"]))

## Step 5: Visualize Activation Norms Across Layers

Four-panel visualization showing how activations behave across all 32 layers.

In [ ]:
layer_names = ["layer_{}".format(i) for i in range(model.config.num_hidden_layers)]
layer_norms = [activation_stats[n]["norm"] for n in layer_names if n in activation_stats]
layer_means = [activation_stats[n]["mean"] for n in layer_names if n in activation_stats]
layer_stds = [activation_stats[n]["std"] for n in layer_names if n in activation_stats]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Llama 3.1 8B Layer Analysis", fontsize=16, fontweight="bold")

# Norms
ax = axes[0, 0]
ax.bar(range(len(layer_norms)), layer_norms, color="steelblue", alpha=0.8)
ax.set_xlabel("Layer Index")
ax.set_ylabel("Activation Norm")
ax.set_title("Activation Norms Across Layers")
ax.grid(axis="y", alpha=0.3)

# Means
ax = axes[0, 1]
ax.plot(range(len(layer_means)), layer_means, "o-", color="coral", markersize=3)
ax.set_xlabel("Layer Index")
ax.set_ylabel("Mean Activation")
ax.set_title("Mean Activation Value per Layer")
ax.grid(alpha=0.3)

# Std dev
ax = axes[1, 0]
ax.plot(range(len(layer_stds)), layer_stds, "s-", color="forestgreen", markersize=3)
ax.set_xlabel("Layer Index")
ax.set_ylabel("Activation Std Dev")
ax.set_title("Activation Standard Deviation per Layer")
ax.grid(alpha=0.3)

# Attention vs MLP norms
num_l = model.config.num_hidden_layers
attn_norms = [activation_stats["layer_{}_attn".format(i)]["norm"] for i in range(num_l) if "layer_{}_attn".format(i) in activation_stats]
mlp_norms = [activation_stats["layer_{}_mlp".format(i)]["norm"] for i in range(num_l) if "layer_{}_mlp".format(i) in activation_stats]
x = range(min(len(attn_norms), len(mlp_norms)))
ax = axes[1, 1]
ax.plot(x, attn_norms[:len(x)], "o-", label="Attention", color="royalblue", markersize=3)
ax.plot(x, mlp_norms[:len(x)], "s-", label="MLP", color="tomato", markersize=3)
ax.set_xlabel("Layer Index")
ax.set_ylabel("Activation Norm")
ax.set_title("Attention vs MLP Activation Norms")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "01_activation_analysis.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: {}/01_activation_analysis.png".format(OUTPUT_DIR))

## Step 6: Bottleneck Detection

Check for potential issues: layers with >50% zero activations (dead neurons) or near-zero standard deviation (vanishing signal).

In [ ]:
bottlenecks = []
for name, stats in activation_stats.items():
    if stats["zeros_pct"] > 50:
        bottlenecks.append((name, "{}% dead".format(stats["zeros_pct"])))
    if stats["std"] < 1e-5:
        bottlenecks.append((name, "near-zero std: {:.2e}".format(stats["std"])))

if bottlenecks:
    print("Potential bottlenecks found:")
    for name, issue in bottlenecks:
        print("  {}: {}".format(name, issue))
else:
    print("No bottlenecks detected -- all layers have healthy activations!")

## Step 7: Layer Timing Profiling

Using **pre-hooks** (start timer) and **forward hooks** (stop timer) together to measure per-layer inference time.

In [ ]:
# Remove old activation hooks
for h in handles:
    h.remove()

# Attach timing hooks
layer_times = {}

def make_timing_hook(name):
    def hook(module, input, output):
        layer_times[name]["end"] = time.perf_counter()
    return hook

def make_pre_timing_hook(name):
    def hook(module, input):
        layer_times[name] = {"start": time.perf_counter(), "end": 0}
    return hook

timing_handles = []
for i, layer in enumerate(model.model.layers):
    name = "layer_{}".format(i)
    h1 = layer.register_forward_pre_hook(make_pre_timing_hook(name))
    h2 = layer.register_forward_hook(make_timing_hook(name))
    timing_handles.extend([h1, h2])

# Run a forward pass for timing
with torch.no_grad():
    _ = model(**inputs)

# Calculate durations
durations = {}
for name, times in layer_times.items():
    if times["end"] > 0:
        durations[name] = (times["end"] - times["start"]) * 1000  # ms

sorted_durations = sorted(durations.items(), key=lambda x: x[1], reverse=True)
total = sum(durations.values())

header = "{:<20} {:>10} {:>12}".format("Layer", "Time (ms)", "% of Total")
print(header)
print("-" * len(header))
for name, dur in sorted_durations[:10]:
    print("{:<20} {:>10.3f} {:>11.1f}%".format(name, dur, dur/total*100))
print("{:<20} {:>10.3f} {:>12}".format("TOTAL", total, "100.0%"))

In [ ]:
# Plot timing
fig, ax = plt.subplots(figsize=(14, 5))
layer_indices = [int(n.split("_")[1]) for n in sorted(durations.keys())]
times_ms = [durations["layer_{}".format(i)] for i in layer_indices]
ax.bar(layer_indices, times_ms, color="mediumpurple", alpha=0.8)
ax.set_xlabel("Layer Index")
ax.set_ylabel("Time (ms)")
ax.set_title("Per-Layer Inference Time: Llama 3.1 8B")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "02_layer_timing.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: {}/02_layer_timing.png".format(OUTPUT_DIR))

In [ ]:
# Clean up timing hooks
for h in timing_handles:
    h.remove()

# Save summary JSON
summary = {
    "model": MODEL_ID,
    "prompt": PROMPT,
    "generated_tokens": new_tokens,
    "generation_time_s": gen_time,
    "tokens_per_second": new_tokens / gen_time,
    "load_time_s": load_time,
    "total_parameters_B": sum(p.numel() for p in model.parameters()) / 1e9,
    "num_layers": model.config.num_hidden_layers,
    "hidden_size": model.config.hidden_size,
    "activation_stats": {
        k: {kk: round(vv, 6) if isinstance(vv, float) else vv for kk, vv in v.items()}
        for k, v in activation_stats.items()
    },
}

with open(os.path.join(OUTPUT_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("Summary saved to {}/summary.json".format(OUTPUT_DIR))
print()
print("DONE. All hook outputs saved to:", OUTPUT_DIR)